# behaviors

> The client runtime: delegated event rules that survive htmx swaps

In [ ]:
#| default_exp behaviors

The registry stores a component's behaviors as records: a selector, an event, a handler body. Here we turn those into running code in the browser. Wiring a listener onto each element as it appears would work for a static page, but in a FastHTML app elements arrive continuously as htmx swaps in fragments, so per-element wiring would mean re-initialising after every swap.

Instead we delegate: every event becomes one document-level listener which, when its event fires anywhere on the page, checks whether the target sits inside an element matching the rule's selector. Matching happens at event time against the live DOM, so elements swapped in later behave the same as those in the original page, with no per-element state and no mutation observer. The rule set is plain data, emitted fresh on each page render, which also lets a hyperelement be redefined in a live notebook.

In [ ]:
from fastcore.utils import *
from answerui.registry import Registry

In [ ]:
from answerui.registry import reg, hypelt
from fastcore.test import *

## The runtime

The script below is the whole runtime. Rules arrive as a list of records, each with a selector, an event name, a capture flag, and a handler function. We group them by event and phase, so each pair gets one listener however many rules share it. On an event, each rule in the group looks for the nearest ancestor of the target (the target included) that matches its selector, and calls the handler with it.

`htmx:load` fires on the root of newly swapped content, and what we want to match is that content: the root itself and anything inside it. A `deep` rule therefore searches downward from the target instead of upward. Walking up would run a handler for an ancestor that was already on the page, as though it had just arrived.

Handlers are async functions, so a failure in one rejects its own promise and leaves the other rules on that event running. Listening in the capture phase is opt-in, for events like `toggle` and `scroll` that don't bubble. The script is installed once per document: a regenerated script reaches a fresh page or preview, and a second copy injected into the same document would stack its listeners on the first.

In [ ]:
_runtime = r"""(() => {
const rules = [%s];
const by = {};
rules.forEach(r => (by[r.ev + '|' + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split('|');
  document.addEventListener(ev, e => {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el => r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === '1'});
}
})();"""

## Entries

Each registry record becomes one object literal in that list. Event names pass through as written, with one convention: a leading `:` means the htmx event, so `':after-request'` listens for `htmx:after-request`. The `:` form is only a shorthand, so `deep` is set for `htmx:load` whichever way it is written. The handler body is inserted verbatim into an async function taking `el` and `e`:

In [ ]:
def _event(ev):
    "The DOM event name: a leading `:` is the htmx shorthand"
    return 'htmx'+ev if ev.startswith(':') else ev

def _entry(r):
    ev = _event(r['ev'])
    deep = ev=='htmx:load'
    return (f'{{sel: {dumps(r["sel"])}, ev: {dumps(ev)}, cap: {dumps(r["cap"])}, '
        f'deep: {dumps(deep)}, async fn(el, e) {{ {r["js"]} }}}}')


## Building the script

A behavior's identity is its selector, its event, and its capture flag, with the `:` shorthand normalised first. Declaring the same identity again replaces the earlier handler: the last declaration in registry order ships and the others are dropped. Re-registering a rule keeps its position, so re-running a component's definition never outranks an application's later override. A replacement also keeps the slot of the handler it replaced, so dispatch order within an event follows first declarations. Only an exact match counts. Two selectors that happen to pick the same elements are two behaviors, and both run, since the registry does not read CSS.

`behaviors` is the script's input, the effective handler for each identity with the key of the rule that declared it, and `build_js` fills the runtime with it:


In [ ]:
@patch
def behaviors(self:Registry):
    "The effective behavior per (selector, event, capture): the last declared in registry order, with the key of the rule that declared it"
    res = {}
    for k,r in self.rules.items():
        for b in r.on: res[(b['sel'], _event(b['ev']), b['cap'])] = dict(b, ev=_event(b['ev']), rule=k)
    return res

@patch
def build_js(self:Registry):
    "The behavior script: the effective behaviors, in the order their identities were first declared"
    return _runtime % ',\n'.join(_entry(b) for b in self.behaviors().values())


As examples, let's make a notice that removes itself after a delay (or when a child marked `dismiss` is clicked), and a menu that focuses its first item when its popover opens. The autodismiss listens for `:load`, which htmx fires for the initial page and for every swapped fragment, but never for content already on the page, so swapping something inside a notice doesn't restart it. Since `toggle` doesn't bubble, the menu rule uses the capture phase. Capture makes the event observable, not exclusive to the matched element: a `details` toggled inside the menu would reach the handler too, so it checks that the target is the menu itself:

In [ ]:
NoticeBox = hypelt('NoticeBox', layer='components', doc='A notice that dismisses itself.', on={
    ('&[autodismiss]', ':load'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove()",
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()"})
reg.rule('menu', sel='[popover][menu]', on={('&', 'toggle', True): r"if (e.target === el && e.newState === 'open') el.querySelector('a, button')?.focus()"})
print('\n'.join(l for l in reg.build_js().splitlines() if 'notice-box' in l or '[menu]' in l))

const rules = [{sel: ":is(notice-box)[autodismiss]", ev: "htmx:load", cap: false, deep: true, async fn(el, e) { await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove() }},
{sel: ":is(notice-box) [dismiss]", ev: "click", cap: false, deep: false, async fn(el, e) { el.closest('notice-box').remove() }},
{sel: ":is([popover][menu])", ev: "toggle", cap: true, deep: false, async fn(el, e) { if (e.target === el && e.newState === 'open') el.querySelector('a, button')?.focus() }}];


The `:load` rule became `htmx:load` with `deep` set, the `toggle` rule carries `cap`, and the click rule is plain. The browser does the grouping into listeners, so the script itself is only the rule list.

In [ ]:
s = reg.build_js()
assert '"htmx:load"' in s and 'deep: true' in s and 'cap: true' in s

An application that wants a confirmation before a notice goes registers the same selector and event through the component. `&` expands exactly as it did in the definition, so the identity matches and the confirming handler replaces the original. The effective behaviors say which rule each handler came from:

In [ ]:
NoticeBox.rule('confirm-dismiss', on={('& [dismiss]', 'click'): r"if (confirm('Dismiss this notice?')) el.closest('notice-box').remove()"})
{k: b['rule'] for k,b in reg.behaviors().items()}

{(':is(notice-box)[autodismiss]', 'htmx:load', False): 'NoticeBox.base',
 (':is(notice-box) [dismiss]', 'click', False): 'NoticeBox.confirm-dismiss',
 (':is([popover][menu])', 'toggle', True): 'menu'}

The script carries one click handler for the dismiss button, the confirming one. `NoticeBox.behaviors` still lists both declarations, because it reports what was declared on the component; `reg.behaviors` reports what ships:

In [ ]:
s = reg.build_js()
test_eq(s.count('":is(notice-box) [dismiss]"'), 1)
assert 'confirm(' in s
NoticeBox.behaviors

[{'sel': ':is(notice-box)[autodismiss]',
  'ev': ':load',
  'js': "await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove()",
  'cap': False},
 {'sel': ':is(notice-box) [dismiss]',
  'ev': 'click',
  'js': "el.closest('notice-box').remove()",
  'cap': False},
 {'sel': ':is(notice-box) [dismiss]',
  'ev': 'click',
  'js': "if (confirm('Dismiss this notice?')) el.closest('notice-box').remove()",
  'cap': False}]

Two guarantees the policy makes: `htmx:load` written in full is the same identity as the `:load` shorthand, and re-registering the component's definition does not displace the application's override:

In [ ]:
NoticeBox = hypelt('NoticeBox', layer='components', doc='A notice that dismisses itself.', on={
    ('&[autodismiss]', 'htmx:load'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove()",
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()"})
test_eq(reg.behaviors()[(':is(notice-box) [dismiss]', 'click', False)]['rule'], 'NoticeBox.confirm-dismiss')
[k for k in reg.behaviors() if k[1]=='htmx:load']

[(':is(notice-box)[autodismiss]', 'htmx:load', False)]

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()